# Latent Diffusion — EMNIST · Hyperparameter Tuning

This notebook is dedicated to **hyperparameter tuning** of the `LatentDenoiser` used for
latent diffusion over the frozen VAE latent space of EMNIST.

We grid-search three hyperparameters — the **learning rate**, the **hidden dimension**
(model capacity) and the **noise scheduler** (linear vs cosine) — then **save the best
configuration** and run a **final training** with it, exporting the loss curve, the DDIM
generation snapshots, class-conditioned samples and an additional diagnostic figure.


## 1. Imports


In [ ]:
# neural network libs
import torch
import torch.nn as nn
import math

# data libs
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

# plot / data libs
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# processing libs
import os
import json
from tqdm import tqdm

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Using device: {device}')

## 2. Configuration


In [ ]:
# ── Paths ──────────────────────────────────────────────────────────
VAE_CKPT         = 'training_checkpoints/vae_best.ckpt'              # frozen VAE (encoder/decoder)
DENOISER_CKPT    = 'training_checkpoints/latent_denoiser_best.ckpt'  # final denoiser checkpoint
BEST_PARAMS_PATH = 'results/best_hyperparameters.json'              # best config found by the search
RESULTS_DIR      = 'results'                                        # where exported figures / CSVs go

# ── Data / final-training config ───────────────────────────────────
BATCH_SIZE         = 64
FINAL_TRAIN_EPOCHS = 30   # epochs for the final training with the best hyperparameters

os.makedirs(RESULTS_DIR, exist_ok=True)

## 3. VAE architecture (frozen)

Architecture copied from `VAE_EMNIST.ipynb`; here it is only used to rebuild the frozen
encoder / decoder from the VAE checkpoint. The VAE is **not** trained in this notebook.


In [ ]:
class ReshapeToImage(nn.Module):
    def forward(self, x):
        return x.view(-1, 1, 28, 28)


# ── 2-block architecture ───────────────────────────────────────────

class Encoder2(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32, 64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(64 * 7 * 7, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(64 * 7 * 7, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder2(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 64 * 7 * 7),
            nn.ReLU(),
            nn.Unflatten(1, (64, 7, 7)),
            nn.ConvTranspose2d(64, 32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,  1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


# ── 3-block architecture ───────────────────────────────────────────

class Encoder3(nn.Module):
    def __init__(self, latent_dim, use_diff_sigma_E):
        super().__init__()
        self.conv = nn.Sequential(
            ReshapeToImage(),
            nn.Conv2d(1,   32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(32,  64, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.Flatten(),
        )
        self.fc_mu     = nn.Linear(128 * 3 * 3, latent_dim)
        logvar_out     = latent_dim if use_diff_sigma_E else 1
        self.fc_logvar = nn.Linear(128 * 3 * 3, logvar_out)

    def forward(self, x):
        h = self.conv(x)
        return self.fc_mu(h), self.fc_logvar(h)


class Decoder3(nn.Module):
    def __init__(self, latent_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(latent_dim, 128 * 3 * 3),
            nn.ReLU(),
            nn.Unflatten(1, (128, 3, 3)),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2,
                               padding=1, output_padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(64,  32, kernel_size=4, stride=2, padding=1),
            nn.ReLU(),
            nn.ConvTranspose2d(32,   1, kernel_size=4, stride=2, padding=1),
            nn.Sigmoid(),
            nn.Flatten(),
        )

    def forward(self, z):
        return self.net(z)


# ── Architecture factory ───────────────────────────────────────────

_ARCHITECTURES = {
    '2blocks': (Encoder2, Decoder2),
    '3blocks': (Encoder3, Decoder3),
}

def get_encoder_decoder(arch, latent_dim, use_diff_sigma_E):
    EncoderCls, DecoderCls = _ARCHITECTURES[arch]
    return EncoderCls(latent_dim, use_diff_sigma_E), DecoderCls(latent_dim)

## 4. Load encoder and decoder from the VAE checkpoint


In [ ]:
ckpt = torch.load(VAE_CKPT, map_location=device)

LATENT_DIM       = ckpt['latent_dim']
USE_DIFF_SIGMA_E = ckpt['use_diff_sigma_E']
ARCH             = ckpt.get('arch', '2blocks')   # backward compat

encoder, decoder = get_encoder_decoder(ARCH, LATENT_DIM, USE_DIFF_SIGMA_E)
encoder = encoder.to(device)
encoder.load_state_dict(ckpt['encoder'])
encoder.eval()
for p in encoder.parameters():
    p.requires_grad_(False)

decoder = decoder.to(device)
decoder.load_state_dict(ckpt['decoder'])
decoder.eval()
for p in decoder.parameters():
    p.requires_grad_(False)

print(f'Loaded VAE checkpoint from epoch {ckpt["epoch"]} '
      f'(test loss {ckpt["test_loss"]:.4f})')
print(f'LATENT_DIM = {LATENT_DIM}, ARCH = {ARCH}')

## 5. Latent Denoiser model


In [ ]:
class LatentDenoiser(nn.Module):
    def __init__(self, latent_dim, num_classes, time_embedding_dim=256, hidden_dim=256, label_embedding_dim=64):
        super().__init__()
        self.latent_dim = latent_dim
        self.time_embedding_dim = time_embedding_dim
        self.hidden_dim = hidden_dim
        self.num_classes = num_classes
        self.label_embedding_dim = label_embedding_dim

        # Time embedding
        self.time_mlp = nn.Sequential(
            nn.Linear(time_embedding_dim, time_embedding_dim * 4),
            nn.GELU(),
            nn.Linear(time_embedding_dim * 4, time_embedding_dim)
        )

        # Label embedding
        self.label_emb = nn.Embedding(num_classes, label_embedding_dim)

        # Input projection + initial layer
        # Now includes latent_dim + time_embedding_dim + label_embedding_dim
        self.proj_in = nn.Linear(latent_dim + time_embedding_dim + label_embedding_dim, hidden_dim)

        # Downsampling blocks with residual connections
        self.down1 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim * 2)
        )
        self.down_res1 = nn.Linear(hidden_dim, hidden_dim * 2) # For skip connection

        self.down2 = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim * 4)
        )
        self.down_res2 = nn.Linear(hidden_dim * 2, hidden_dim * 4) # For skip connection

        # Bottleneck
        self.mid = nn.Sequential(
            nn.Linear(hidden_dim * 4, hidden_dim * 4),
            nn.GELU(),
            nn.Linear(hidden_dim * 4, hidden_dim * 4)
        )

        # Upsampling blocks with residual connections (from skip + current layer)
        # Input is concat of mid output and skip from down2
        self.up1 = nn.Sequential(
            nn.Linear(hidden_dim * 4 + hidden_dim * 4, hidden_dim * 2),
            nn.GELU(),
            nn.Linear(hidden_dim * 2, hidden_dim * 2)
        )
        self.up_res1 = nn.Linear(hidden_dim * 4 + hidden_dim * 4, hidden_dim * 2) # For skip connection

        # Input is concat of up1 output and skip from down1
        self.up2 = nn.Sequential(
            nn.Linear(hidden_dim * 2 + hidden_dim * 2, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim)
        )
        self.up_res2 = nn.Linear(hidden_dim * 2 + hidden_dim * 2, hidden_dim) # For skip connection

        # Output layer
        self.proj_out = nn.Linear(hidden_dim, latent_dim)

    # Added gelu activation here, as it was missing in the original code for some residual paths
    def gelu(self, x):
        return nn.functional.gelu(x)

    def forward(self, z, t, labels): # Added labels as input
        # z: latent vector (batch_size, latent_dim)
        # t: time step (batch_size,)
        # labels: class labels (batch_size,)

        # Create time embedding
        t_embed = self.time_embedding(t)
        t_embed = self.time_mlp(t_embed)

        # Create label embedding
        label_embed = self.label_emb(labels)

        # Concatenate latent vector, time embedding, and label embedding
        x = torch.cat([z, t_embed, label_embed], dim=1)
        x = self.proj_in(x) # (B, hidden_dim)

        # Downsampling path
        # Block 1
        h1 = x
        x = self.down1(x) + self.down_res1(h1) # Residual connection
        x = self.gelu(x)
        skip1 = x # Store for skip connection

        # Block 2
        h2 = x
        x = self.down2(x) + self.down_res2(h2) # Residual connection
        x = self.gelu(x)
        skip2 = x # Store for skip connection

        # Bottleneck
        x = self.mid(x) + x # Residual connection
        x = self.gelu(x) # Added missing activation here

        # Upsampling path (with skip connections)
        # Block 1
        x_up1_input = torch.cat([x, skip2], dim=1) # Concatenate with skip from down2
        h3 = x_up1_input
        x = self.up1(x_up1_input) + self.up_res1(h3) # Residual connection
        x = self.gelu(x)

        # Block 2
        x_up2_input = torch.cat([x, skip1], dim=1) # Concatenate with skip from down1
        h4 = x_up2_input
        x = self.up2(x_up2_input) + self.up_res2(h4) # Residual connection
        x = self.gelu(x)

        # Output
        x = self.proj_out(x)

        return x

    def time_embedding(self, t):
        # Sinusoidal positional embeddings for time
        half_dim = self.time_embedding_dim // 2
        emb = torch.log(torch.tensor(10000.0)) / (half_dim - 1)
        emb = torch.exp(torch.arange(half_dim, device=t.device) * -emb)
        emb = t[:, None] * emb[None, :] # Reshape t to (batch_size, 1) for broadcasting
        emb = torch.cat((emb.sin(), emb.cos()), dim=-1)
        return emb

# EMNIST 'byclass' split has 62 classes (0-9, A-Z, a-z)
NUM_EMNIST_CLASSES = 62
print(f'LatentDenoiser defined. EMNIST classes: {NUM_EMNIST_CLASSES}.')

### 5.1 Noise schedules and diffusion helpers


In [ ]:
def linear_beta_schedule(timesteps):
    scale = 1000 / timesteps
    beta_start = scale * 0.0001
    beta_end = scale * 0.02
    # Changed dtype to torch.float32 to match model's expected input
    return torch.linspace(beta_start, beta_end, timesteps, dtype=torch.float32)

def cosine_beta_schedule(timesteps, s=0.008):
    """cosine schedule as proposed in https://arxiv.org/abs/2102.09672"""
    steps = timesteps + 1
    x = torch.linspace(0, timesteps, steps, dtype=torch.float32) # Changed dtype to float32
    alphas_cumprod = torch.cos(((x / timesteps) + s) / (1 + s) * math.pi * 0.5) ** 2
    alphas_cumprod = alphas_cumprod / alphas_cumprod[0]
    betas = 1 - (alphas_cumprod[1:] / alphas_cumprod[:-1])
    return torch.clip(betas, 0.0001, 0.9999)

TIMESTEPS = 1000  # Number of diffusion steps - Standardized to 1000 for better results
b_t = linear_beta_schedule(TIMESTEPS).to(device) # Or use cosine_beta_schedule

a_t = 1. - b_t
alpha_bar_t = torch.cumprod(a_t, dim=0)

# Helper functions for diffusion process
def extract(a, t, x_shape):
    batch_size = t.shape[0]
    out = a.gather(-1, t).to(t.device)
    # Explicitly reshape to (batch_size, 1) to ensure broadcasting with latent vectors
    return out.view(batch_size, 1)

def get_noisy_latent(x_start, t, alpha_bar_t, device):
    # Forward diffusion (sample from q(z_t | z_0))
    sqrt_alpha_bar_t = extract(torch.sqrt(alpha_bar_t), t, x_start.shape)
    sqrt_one_minus_alpha_bar_t = extract(torch.sqrt(1. - alpha_bar_t), t, x_start.shape)
    epsilon = torch.randn_like(x_start, device=device)
    noisy_latent = sqrt_alpha_bar_t * x_start + sqrt_one_minus_alpha_bar_t * epsilon
    return noisy_latent, epsilon

print(f'Noise schedule (betas) initialized with {TIMESTEPS} timesteps.')

## 6. Training data (EMNIST)

We load the EMNIST **train** split (the same split the VAE was trained on) and define the
shared MSE loss. Every grid-search experiment and the final training use this
`train_loader`.


In [ ]:
# EMNIST split is stored in the VAE checkpoint (fallback: 'byclass').
EMNIST_SPLIT = ckpt.get('emnist_split', 'byclass')

transform = transforms.ToTensor()
train_dataset = datasets.EMNIST(
    root='data', split=EMNIST_SPLIT, train=True,
    transform=transform, download=True,
)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)

# Loss shared by every experiment and the final training:
# MSE between predicted and true noise.
loss_fn = nn.MSELoss()

print(f"EMNIST '{EMNIST_SPLIT}' train split loaded: {len(train_dataset)} samples, "
      f"batch size {BATCH_SIZE}.")

## 7. Hyperparameter Search for the Latent Denoiser

We grid-search three hyperparameters, training a **fresh** `LatentDenoiser` for every
combination:

- **learning rate** — `learning_rates`
- **hidden dimension** (model capacity) — `hidden_dims`
- **noise scheduler** — `schedulers` (linear vs cosine)

Early stopping is kept with a **fixed** patience (a training-control knob, not a model
hyperparameter, so it is not part of the grid). Each experiment is scored by its best
(early-stopped) average training loss.


In [ ]:
class EarlyStopping:
    """Early stops the training if validation loss doesn't improve after a given patience."""
    def __init__(self, patience=5, verbose=False, delta=0, path='checkpoint.pt', trace_func=print):
        """
        Args:
            patience (int): How long to wait after last time validation loss improved.
                            Default: 7
            verbose (bool): If True, prints a message for each validation loss improvement.
                            Default: False
            delta (float): Minimum change in the monitored quantity to qualify as an improvement.
                           Default: 0
            path (str): Path for the checkpoint to be saved to.
                        Default: 'checkpoint.pt'
            trace_func (function): trace print function.
                                   Default: print
        """
        self.patience = patience
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.val_loss_min = np.inf # Changed from np.Inf to np.inf
        self.delta = delta
        self.path = path
        self.trace_func = trace_func

    def __call__(self, val_loss, model):

        score = -val_loss

        if self.best_score is None:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
        elif score < self.best_score + self.delta:
            self.counter += 1
            if self.verbose:
                self.trace_func(f'EarlyStopping counter: {self.counter} out of {self.patience}')
            if self.counter >= self.patience:
                self.early_stop = True
        else:
            self.best_score = score
            self.save_checkpoint(val_loss, model)
            self.counter = 0

    def save_checkpoint(self, val_loss, model):
        '''Saves model when validation loss decrease.'''
        if self.verbose:
            self.trace_func(f'Validation loss decreased ({self.val_loss_min:.6f} --> {val_loss:.6f}).  Saving model ...')
        os.makedirs(os.path.dirname(self.path), exist_ok=True) # Ensure directory exists
        torch.save(model.state_dict(), self.path)
        self.val_loss_min = val_loss

EARLY_STOPPING_PATH = 'training_checkpoints/latent_denoiser_early_stop.ckpt'
print(f"EarlyStopping class and EARLY_STOPPING_PATH='{EARLY_STOPPING_PATH}' defined.")

In [ ]:
# Define the hyperparameter search space
learning_rates = [1e-4, 5e-5, 1e-3]
hidden_dims = [LATENT_DIM * 2, LATENT_DIM * 3, LATENT_DIM * 4] # Explore different model capacities
schedulers = ['linear', 'cosine'] # Noise schedule to compare

# Early stopping patience is fixed (no longer part of the grid)
FIXED_PATIENCE = 5

# Max epochs for each experiment (early stopping will likely stop it earlier)
GRID_SEARCH_MAX_EPOCHS = 50 # Set a reasonable upper bound for each experiment

# Directory to save checkpoints for each grid search experiment
GRID_SEARCH_CHECKPOINTS_DIR = 'grid_search_checkpoints'
os.makedirs(GRID_SEARCH_CHECKPOINTS_DIR, exist_ok=True)

print(f"Grid search parameters defined. Checkpoints will be saved in: {GRID_SEARCH_CHECKPOINTS_DIR}")

In [ ]:
import pandas as pd

def get_alpha_bar(scheduler, timesteps, device):
    """Compute the cumulative product of alphas for the chosen noise scheduler."""
    if scheduler == 'linear':
        betas = linear_beta_schedule(timesteps)
    elif scheduler == 'cosine':
        betas = cosine_beta_schedule(timesteps)
    else:
        raise ValueError(f"Unknown scheduler: {scheduler}")
    betas = betas.to(device)
    alphas = 1. - betas
    return torch.cumprod(alphas, dim=0)


def train_denoiser_experiment(lr, hidden_d, scheduler, max_epochs, experiment_name):
    print(f"\n--- Running experiment: {experiment_name} ---")
    print(f"  LR={lr}, HiddenDim={hidden_d}, Scheduler={scheduler}, MaxEpochs={max_epochs}")

    # Noise schedule for this experiment
    exp_alpha_bar_t = get_alpha_bar(scheduler, TIMESTEPS, device)

    # Re-initialize model for each experiment
    current_latent_denoiser = LatentDenoiser(LATENT_DIM, num_classes=NUM_EMNIST_CLASSES, hidden_dim=hidden_d).to(device)
    current_optimizer = torch.optim.Adam(current_latent_denoiser.parameters(), lr=lr)

    # Define a unique path for this experiment's checkpoint
    exp_ckpt_path = os.path.join(GRID_SEARCH_CHECKPOINTS_DIR, f"{experiment_name}.ckpt")
    current_early_stopper = EarlyStopping(patience=FIXED_PATIENCE, verbose=False, path=exp_ckpt_path)

    experiment_training_losses = []
    epochs_ran = 0
    best_loss_for_experiment = float('inf')

    for epoch in range(max_epochs):
        current_latent_denoiser.train()
        total_loss = 0
        # Use tqdm for progress bar within each epoch
        for batch_idx, (images, labels) in enumerate(tqdm(train_loader, desc=f"  Epoch {epoch+1}/{max_epochs}")):
            x_0 = images.view(images.size(0), -1).to(device)
            labels = labels.to(device)

            with torch.no_grad(): # VAE encoder is frozen
                mu_0, _ = encoder(x_0)

            t = torch.randint(0, TIMESTEPS, (mu_0.shape[0],), device=device).long()
            noisy_latent, epsilon_true = get_noisy_latent(mu_0, t, exp_alpha_bar_t, device)
            epsilon_pred = current_latent_denoiser(noisy_latent, t, labels)

            loss = loss_fn(epsilon_pred, epsilon_true)

            current_optimizer.zero_grad()
            loss.backward()
            current_optimizer.step()

            total_loss += loss.item()

        avg_loss = total_loss / len(train_loader)
        experiment_training_losses.append(avg_loss)
        print(f"  Experiment {experiment_name} - Epoch {epoch+1} finished, Average Loss: {avg_loss:.4f}")

        current_early_stopper(avg_loss, current_latent_denoiser)
        if current_early_stopper.early_stop:
            print(f"  Early stopping triggered for {experiment_name} after {epoch+1} epochs.")
            epochs_ran = epoch + 1
            break

        epochs_ran = epoch + 1 # Update epochs_ran if not early stopped

    # Load the best model's state dict to get the true best loss from early stopping
    if os.path.exists(exp_ckpt_path) and current_early_stopper.val_loss_min != float('inf'): # Check if a checkpoint was actually saved
        # No need to load state_dict here, the best_score from early stopper already holds the minimum loss value
        best_loss_for_experiment = current_early_stopper.val_loss_min
    else:
        # If no checkpoint was saved (e.g., training was very short or no improvement), use the last loss
        if experiment_training_losses:
            best_loss_for_experiment = experiment_training_losses[-1]
        else:
            best_loss_for_experiment = float('inf') # No training happened

    return {
        'experiment_name': experiment_name,
        'learning_rate': lr,
        'hidden_dim': hidden_d,
        'scheduler': scheduler,
        'max_epochs_configured': max_epochs,
        'epochs_ran': epochs_ran,
        'best_loss': best_loss_for_experiment,
        'checkpoint_path': exp_ckpt_path
    }

In [ ]:
grid_search_results = []
experiment_count = 0

for lr in learning_rates:
    for hd in hidden_dims:
        for sched in schedulers:
            experiment_count += 1
            experiment_name = f"exp_{experiment_count:03d}_lr{lr}_hd{hd}_{sched}"

            result = train_denoiser_experiment(lr, hd, sched, GRID_SEARCH_MAX_EPOCHS, experiment_name)
            grid_search_results.append(result)

print("\nGrid search complete. All experiments have been run.")

In [ ]:
# Display grid search results
results_df = pd.DataFrame(grid_search_results)
display(results_df.sort_values(by='best_loss').head(10))

# Optionally save results to a CSV
results_csv_path = os.path.join(RESULTS_DIR, 'latent_denoiser_grid_search_results.csv')
results_df.to_csv(results_csv_path, index=False)
print(f"Full grid search results saved to '{results_csv_path}'.")

In [ ]:
# Visualize the grid search: best loss for every (lr, hidden_dim, scheduler) combination.
from matplotlib.patches import Patch

plot_df = results_df.sort_values('best_loss').reset_index(drop=True)
labels  = [f"lr={r.learning_rate}\nhd={r.hidden_dim}\n{r.scheduler}"
           for r in plot_df.itertuples()]
colors  = ['tab:blue' if s == 'linear' else 'tab:orange' for s in plot_df['scheduler']]

plt.figure(figsize=(max(10, len(plot_df) * 0.7), 6))
plt.bar(range(len(plot_df)), plot_df['best_loss'], color=colors)
plt.xticks(range(len(plot_df)), labels, fontsize=7)
plt.ylabel('Best (early-stopped) training loss')
plt.title('Hyperparameter search — best loss per configuration (sorted)')
plt.grid(axis='y', alpha=0.3)
plt.legend(handles=[Patch(color='tab:blue', label='linear'),
                    Patch(color='tab:orange', label='cosine')], title='scheduler')
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'grid_search_results.png'), dpi=200)
plt.show()
print(f"Grid search results plot saved to '{os.path.join(RESULTS_DIR, 'grid_search_results.png')}'.")

## 8. Select and save the best hyperparameters


In [ ]:
# Pick the configuration with the lowest best_loss and persist it to disk.
best_row = results_df.sort_values('best_loss').iloc[0]

best_params = {
    'learning_rate': float(best_row['learning_rate']),
    'hidden_dim':    int(best_row['hidden_dim']),
    'scheduler':     str(best_row['scheduler']),
    'best_loss':     float(best_row['best_loss']),
}

with open(BEST_PARAMS_PATH, 'w') as f:
    json.dump(best_params, f, indent=2)

print('Best hyperparameters found:')
for k, v in best_params.items():
    print(f'  {k}: {v}')
print(f"\nSaved to '{BEST_PARAMS_PATH}'.")

## 9. Final training with the best hyperparameters

We reload the best configuration, rebuild the noise schedule to match the chosen
scheduler, and train a **fresh** `LatentDenoiser` for `FINAL_TRAIN_EPOCHS` epochs.


In [ ]:
# Load the best hyperparameters (from disk so this section can be run standalone).
with open(BEST_PARAMS_PATH) as f:
    best_params = json.load(f)

best_lr         = best_params['learning_rate']
best_hidden_dim = best_params['hidden_dim']
best_scheduler  = best_params['scheduler']
print(f"Training final model with lr={best_lr}, hidden_dim={best_hidden_dim}, "
      f"scheduler='{best_scheduler}'.")

# Rebuild the noise schedule for the CHOSEN scheduler and update the global tensors
# (b_t / alpha_bar_t) so the sampling functions further down use the matching schedule.
if best_scheduler == 'cosine':
    b_t = cosine_beta_schedule(TIMESTEPS).to(device)
else:
    b_t = linear_beta_schedule(TIMESTEPS).to(device)
a_t = 1. - b_t
alpha_bar_t = torch.cumprod(a_t, dim=0)

# Fresh model + optimizer with the best hyperparameters.
latent_denoiser = LatentDenoiser(
    LATENT_DIM, num_classes=NUM_EMNIST_CLASSES, hidden_dim=best_hidden_dim
).to(device)
optimizer = torch.optim.Adam(latent_denoiser.parameters(), lr=best_lr)
print(f'Latent Denoiser created with '
      f'{sum(p.numel() for p in latent_denoiser.parameters())} parameters.')

In [ ]:
print("Starting final training of the Latent Denoiser...")
training_losses = []

for epoch in range(FINAL_TRAIN_EPOCHS):
    latent_denoiser.train()
    total_loss = 0
    for images, labels in tqdm(train_loader, desc=f"Epoch {epoch+1}/{FINAL_TRAIN_EPOCHS}"):
        x_0 = images.view(images.size(0), -1).to(device)   # original images
        labels = labels.to(device)

        with torch.no_grad():                              # VAE encoder is frozen
            mu_0, _ = encoder(x_0)

        # Sample a time step per latent and add noise (forward diffusion).
        t = torch.randint(0, TIMESTEPS, (mu_0.shape[0],), device=device).long()
        noisy_latent, epsilon_true = get_noisy_latent(mu_0, t, alpha_bar_t, device)

        # Predict the noise (conditioned on the class label) and optimize.
        epsilon_pred = latent_denoiser(noisy_latent, t, labels)
        loss = loss_fn(epsilon_pred, epsilon_true)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(train_loader)
    training_losses.append(avg_loss)
    print(f"Epoch {epoch+1} finished, Average Loss: {avg_loss:.4f}")

print("Final training complete.")

In [ ]:
# Save the final model checkpoint (records the best hyperparameters used).
torch.save({
    'epoch': FINAL_TRAIN_EPOCHS,
    'model_state_dict': latent_denoiser.state_dict(),
    'optimizer_state_dict': optimizer.state_dict(),
    'final_loss': training_losses[-1],
    'latent_dim': LATENT_DIM,
    'num_emnist_classes': NUM_EMNIST_CLASSES,
    'hidden_dim': latent_denoiser.hidden_dim,
    'time_embedding_dim': latent_denoiser.time_embedding_dim,
    'label_embedding_dim': latent_denoiser.label_embedding_dim,
    'timesteps': TIMESTEPS,
    'scheduler': best_scheduler,
    'learning_rate': best_lr,
    'training_losses': training_losses,
}, DENOISER_CKPT)
print(f"Final Latent Denoiser saved to '{DENOISER_CKPT}'.")

## 10. Results and visualizations

Using the final model we export four figures to `results/`:

1. **Loss curve** — `diffusion_loss_curve.png`
2. **DDIM generation snapshots** — `diffusion_generation_process.png`
3. **Class-conditioned samples** (one per EMNIST class) — `diffusion_samples_all_classes.png`
4. **Unconditioned samples** (diversity check) — `diffusion_samples.png`


In [ ]:
@torch.no_grad()
def p_sample(model, x, t, t_index, betas, labels): # Added labels as input
    # Deterministic reverse diffusion step (DDIM, eta=0 / slide eq. 25):
    # no noise is injected — z[k] is obtained deterministically from z[k+1]
    # and the predicted x_0, unlike the stochastic DDPM ancestral sampler.

    sqrt_alphas_cumprod_t = extract(torch.sqrt(alpha_bar_t), t, x.shape)
    sqrt_one_minus_alphas_cumprod_t = extract(torch.sqrt(1. - alpha_bar_t), t, x.shape)

    # Predict noise
    pred_noise = model(x, t, labels) # Pass labels to the model

    # Predict x_0 from x_t and the predicted noise
    x_0_pred = (x - sqrt_one_minus_alphas_cumprod_t * pred_noise) / sqrt_alphas_cumprod_t

    # t_index == 0 means we've reached the data (z_0) estimate directly
    if t_index == 0:
        return x_0_pred

    # Deterministic jump to t-1 using the predicted x_0 and predicted noise
    t_prev = t - 1
    alpha_bar_t_prev = extract(alpha_bar_t, t_prev, x.shape)
    x_prev = torch.sqrt(alpha_bar_t_prev) * x_0_pred + torch.sqrt(1. - alpha_bar_t_prev) * pred_noise

    return x_prev

@torch.no_grad()
def p_sample_loop(model, shape, timesteps, device, labels): # Added labels as input
    # Full sampling loop
    img = torch.randn(shape, device=device) # Start with pure noise

    for i in tqdm(reversed(range(0, timesteps)), desc='Sampling', total=timesteps):
        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        img = p_sample(model, img, t, i, b_t, labels) # Pass labels to p_sample

    return img

@torch.no_grad()
def generate_samples(model, vae_decoder, num_samples, latent_dim, timesteps, device, conditioning_labels=None): # Added conditioning_labels
    model.eval()
    vae_decoder.eval()

    if conditioning_labels is None:
        # If no specific labels are provided, generate random labels for sampling
        conditioning_labels = torch.randint(0, NUM_EMNIST_CLASSES, (num_samples,), device=device, dtype=torch.long)
    elif isinstance(conditioning_labels, int):
        # If a single label is provided, create a batch of that label
        conditioning_labels = torch.full((num_samples,), conditioning_labels, device=device, dtype=torch.long)
    else:
        # Ensure conditioning_labels are on the correct device and type
        conditioning_labels = conditioning_labels.to(device).long()
        if conditioning_labels.shape[0] != num_samples:
            raise ValueError(f"Number of conditioning labels ({conditioning_labels.shape[0]}) must match num_samples ({num_samples})")

    # Sample latent vectors from the diffusion model
    sampled_latents = p_sample_loop(model, (num_samples, latent_dim), timesteps, device, conditioning_labels) # Pass conditioning_labels

    # Decode latents into images using the VAE decoder
    generated_images = vae_decoder(sampled_latents)

    return generated_images

print("Sampling functions 'p_sample', 'p_sample_loop', and 'generate_samples' defined (deterministic DDIM-style sampler).")

In [ ]:
def fix_emnist_image(img: np.ndarray) -> np.ndarray:
    """Undo EMNIST's built-in 90 deg CW rotation + horizontal mirror (visualization only)."""
    return np.rot90(np.fliplr(img))

def get_emnist_char(class_id):
    """Map an EMNIST 'byclass' label id to its character."""
    if 0 <= class_id <= 9:
        return str(class_id)
    elif 10 <= class_id <= 35:
        return chr(ord('A') + class_id - 10)
    elif 36 <= class_id <= 61:
        return chr(ord('a') + class_id - 36)
    return '?'

print("Visualization helpers 'fix_emnist_image' and 'get_emnist_char' defined.")

### 10.1 Loss curve


In [ ]:
plt.figure(figsize=(10, 6))
plt.plot(range(1, len(training_losses) + 1), training_losses, marker='o', linestyle='-')
plt.title('Latent Denoiser Training Loss per Epoch (final model)')
plt.xlabel('Epoch')
plt.ylabel('Average MSE Loss')
plt.grid(True)
plt.xticks(range(1, len(training_losses) + 1))
plt.tight_layout()
plt.savefig(os.path.join(RESULTS_DIR, 'diffusion_loss_curve.png'), dpi=200)
plt.show()
print(f"Loss curve saved to '{os.path.join(RESULTS_DIR, 'diffusion_loss_curve.png')}'.")

### 10.2 DDIM generation snapshots


In [ ]:
@torch.no_grad()
def p_sample_loop_with_snapshots(model, shape, timesteps, device, labels, snapshot_timesteps):
    """Same as p_sample_loop, but also returns decoded latents at the requested timesteps.

    `img` entering iteration i holds x_i; after p_sample it holds x_{i-1}
    (or the final x_0 estimate when i == 0), so snapshots are recorded
    accordingly to keep the t= label accurate.
    """
    img = torch.randn(shape, device=device)
    snapshots = {}  # t -> latent tensor (cloned)

    for i in tqdm(reversed(range(0, timesteps)), desc='Sampling (with snapshots)', total=timesteps):
        if i in snapshot_timesteps:
            snapshots[i] = img.clone()  # state entering this step = x_i

        t = torch.full((shape[0],), i, device=device, dtype=torch.long)
        img = p_sample(model, img, t, i, b_t, labels)

        if i == 0:
            snapshots[0] = img.clone()  # final generated latent = x_0

    return img, snapshots


latent_denoiser.eval()

NUM_SNAPSHOT_SAMPLES = 6
snapshot_labels = torch.randint(0, NUM_EMNIST_CLASSES, (NUM_SNAPSHOT_SAMPLES,), device=device, dtype=torch.long)

# Evenly spaced timesteps from T-1 (pure noise) down to 0 (final sample)
N_SNAPSHOTS = 8
snapshot_timesteps = sorted(set(
    int(round(x)) for x in np.linspace(TIMESTEPS - 1, 0, N_SNAPSHOTS)
), reverse=True)

_, snapshots = p_sample_loop_with_snapshots(
    latent_denoiser, (NUM_SNAPSHOT_SAMPLES, LATENT_DIM), TIMESTEPS, device,
    snapshot_labels, snapshot_timesteps,
)

# Decode every snapshot back to image space
decoded_snapshots = {
    t: decoder(z).cpu().view(NUM_SNAPSHOT_SAMPLES, 28, 28).numpy()
    for t, z in snapshots.items()
}

ordered_ts = sorted(decoded_snapshots.keys(), reverse=True)

fig, axes = plt.subplots(NUM_SNAPSHOT_SAMPLES, len(ordered_ts),
                          figsize=(len(ordered_ts) * 1.6, NUM_SNAPSHOT_SAMPLES * 1.6))
for row in range(NUM_SNAPSHOT_SAMPLES):
    for col, t in enumerate(ordered_ts):
        img = fix_emnist_image(decoded_snapshots[t][row])
        ax = axes[row, col]
        ax.imshow(img, cmap='gray')
        ax.axis('off')
        if row == 0:
            ax.set_title(f't={t}', fontsize=9)

plt.suptitle('Snapshots of the Deterministic Diffusion (DDIM) Generation Process', fontsize=14)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])

os.makedirs(RESULTS_DIR, exist_ok=True)
plt.savefig(os.path.join(RESULTS_DIR, 'diffusion_generation_process.png'), dpi=200)
plt.show()

print(f"Saved generation-process snapshots to "
      f"'{os.path.join(RESULTS_DIR, 'diffusion_generation_process.png')}'.")

### 10.3 Class-conditioned samples (one per EMNIST class)


In [ ]:
# One class-conditioned sample for every EMNIST class (0..61).
latent_denoiser.eval()
NUM_SAMPLES_TO_GENERATE = NUM_EMNIST_CLASSES
conditioning_labels = torch.arange(NUM_EMNIST_CLASSES, device=device, dtype=torch.long)

generated_images_conditioned = generate_samples(
    latent_denoiser, decoder, NUM_SAMPLES_TO_GENERATE, LATENT_DIM, TIMESTEPS, device, conditioning_labels
)

n_cols = 9
n_rows = (NUM_EMNIST_CLASSES + n_cols - 1) // n_cols
fig, axes = plt.subplots(n_rows, n_cols, figsize=(n_cols * 2, n_rows * 2))
axes = axes.flatten()

for i in range(NUM_SAMPLES_TO_GENERATE):
    img_fixed = fix_emnist_image(generated_images_conditioned[i].cpu().view(28, 28).numpy())
    ax = axes[i]
    ax.imshow(img_fixed, cmap='gray')
    ax.axis('off')
    ax.set_title(f'Cond: {get_emnist_char(conditioning_labels[i].item())}', fontsize=8)

for j in range(NUM_SAMPLES_TO_GENERATE, len(axes)):
    axes[j].axis('off')

plt.suptitle('Generated Samples (Latent Diffusion with Conditioning) - All EMNIST Classes', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(RESULTS_DIR, 'diffusion_samples_all_classes.png'), dpi=200)
plt.show()
print(f"Saved conditioned samples to '{os.path.join(RESULTS_DIR, 'diffusion_samples_all_classes.png')}'.")

### 10.4 Unconditioned samples (diversity check)


In [ ]:
# Unconditioned (random-class) samples - a quick diversity check of the final model.
NUM_SAMPLES_TO_GENERATE = 16
latent_denoiser.eval()

generated_images = generate_samples(
    latent_denoiser, decoder, NUM_SAMPLES_TO_GENERATE, LATENT_DIM, TIMESTEPS, device
)

fig, axes = plt.subplots(4, 4, figsize=(8, 8))
for i, ax in enumerate(axes.flatten()):
    img = fix_emnist_image(generated_images[i].cpu().view(28, 28).numpy())
    ax.imshow(img, cmap='gray')
    ax.axis('off')
plt.suptitle(f'Unconditioned Generated Samples (after {FINAL_TRAIN_EPOCHS} epochs)', fontsize=16)
plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.savefig(os.path.join(RESULTS_DIR, 'diffusion_samples.png'), dpi=200)
plt.show()
print(f"Saved unconditioned samples to '{os.path.join(RESULTS_DIR, 'diffusion_samples.png')}'.")